## Stats for Token Length
Here we will try to understand distribution of proteome lengths in tokens so we can choose sensible values for stuff like the following

- max chunk size (context window for training)
- overlap b/n chunks
- batch sizes (memory estimate)

In [1]:
import torch
import pandas as pd
from tqdm.auto import tqdm
from protein_tokenizer import ProteinTokenizer

/Users/rishi/Documents/phylogen/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
CSV_PATH = "../../data/ecoli_processed_pairs/ecoli_pairs_concat.csv"
df = pd.read_csv(CSV_PATH, dtype={'genome_id': str})
df.head()

,genome_id,antibiotic,resistant_phenotype,genome_name,taxon_id,taxon_lineage_names,genome_length,antimicrobial_resistance,antimicrobial_resistance_evidence,mutated_protein_path,unmutated_protein_path,reversions_applied,reversion_count,mutated_proteome,unmutated_proteome
0,562.8523,ciprofloxacin,Resistant,Escherichia coli,562,cellular organisms::Bacteria::Pseudomonadati::...,5235035,NaN,NaN,ecoli_processed_pairs/562.8523_mutated_protein...,ecoli_processed_pairs/562.8523_unmutated_prote...,True,3,<PROT>MIKFSATLLATLIAASVNAATVDLRIMETTDLHSNMMDFD...,<PROT>MIKFSATLLATLIAASVNAATVDLRIMETTDLHSNMMDFD...
1,562.7542,ciprofloxacin,Resistant,Escherichia coli,562,cellular organisms::Bacteria::Pseudomonadati::...,5589103,NaN,NaN,ecoli_processed_pairs/562.7542_mutated_protein...,ecoli_processed_pairs/562.7542_unmutated_prote...,True,4,<PROT>MATKTTTAPETDSKRTQLFLQSVSIGQNEIPREMIVGCTY...,<PROT>MATKTTTAPETDSKRTQLFLQSVSIGQNEIPREMIVGCTY...
2,562.98417,ciprofloxacin,Resistant,Escherichia coli 99aee782-7bb9-11e9-a8d3-68b59...,562,cellular organisms::Bacteria::Pseudomonadati::...,5031625,Resistant::Susceptible,AMR Panel,ecoli_processed_pairs/562.98417_mutated_protei...,ecoli_processed_pairs/562.98417_unmutated_prot...,True,3,<PROT>MLVAQHTVYFPDAFLTQMREAMPSTLSFDDFLAACQRPLR...,<PROT>MLVAQHTVYFPDAFLTQMREAMPSTLSFDDFLAACQRPLR...
3,562.140842,ciprofloxacin,Resistant,Escherichia coli 018-E1-S06,562,cellular organisms::Bacteria::Pseudomonadati::...,4912847,Resistant::Intermediate::Susceptible,AMR Panel,ecoli_processed_pairs/562.140842_mutated_prote...,ecoli_processed_pairs/562.140842_unmutated_pro...,True,3,<PROT>MRVLLFLLLSLFMLSAFSADNLLRWHDAQHYTVQASTPLK...,<PROT>MRVLLFLLLSLFMLSAFSADNLLRWHDAQHYTVQASTPLK...
4,562.135581,ciprofloxacin,Resistant,Escherichia coli 20200203_MGL_06,562,cellular organisms::Bacteria::Pseudomonadati::...,5058347,NaN,NaN,ecoli_processed_pairs/562.135581_mutated_prote...,ecoli_processed_pairs/562.135581_unmutated_pro...,True,3,<PROT>MLDLFADAEPWQEPLAAGAVILRRFAFNAAEQLIRDINDV...,<PROT>MLDLFADAEPWQEPLAAGAVILRRFAFNAAEQLIRDINDV...


In [3]:
tokenizer = ProteinTokenizer.load("tokenizer.json")

In [4]:
lengths = []

for idx, row in tqdm(df.head(20).iterrows(), total=20, desc="Tokenizing test set"):
    proteome_str = row['unmutated_proteome']
    token_ids = tokenizer.encode_fast(proteome_str, add_special_tokens=True, conditioning=None)
    lengths.append(len(token_ids))

Tokenizing test set: 100%|██████████| 20/20 [00:03<00:00,  5.85it/s]


In [5]:
length_series = pd.Series(lengths)

print("\nToken length statistics (including BOS + EOS):")
print(length_series.describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))

print("\nKey percentiles:")
print(f"  Min:       {length_series.min():>6}")
print(f"  25th %:    {length_series.quantile(0.25):>6.0f}")
print(f"  Median:    {length_series.median():>6.0f}")
print(f"  75th %:    {length_series.quantile(0.75):>6.0f}")
print(f"  90th %:    {length_series.quantile(0.90):>6.0f}")
print(f"  95th %:    {length_series.quantile(0.95):>6.0f}")
print(f"  99th %:    {length_series.quantile(0.99):>6.0f}")
print(f"  Max:       {length_series.max():>6}")

chunk_size_candidates = [512, 1024, 2048]
for cs in chunk_size_candidates:
    avg_chunks = length_series.mean() / cs
    print(f"At {cs} tokens/chunk: ~{avg_chunks:.1f} chunks per proteome on average")


Token length statistics (including BOS + EOS):
count    2.000000e+01
mean     1.519401e+06
std      6.304214e+04
min      1.408853e+06
25%      1.476369e+06
50%      1.521034e+06
75%      1.563894e+06
90%      1.567031e+06
95%      1.572038e+06
99%      1.647483e+06
max      1.666344e+06
dtype: float64

Key percentiles:
  Min:       1408853
  25th %:    1476369
  Median:    1521034
  75th %:    1563894
  90th %:    1567031
  95th %:    1572038
  99th %:    1647483
  Max:       1666344
At 512 tokens/chunk: ~2967.6 chunks per proteome on average
At 1024 tokens/chunk: ~1483.8 chunks per proteome on average
At 2048 tokens/chunk: ~741.9 chunks per proteome on average
